# Day 08 — Hybrid search and rank fusion: two retrievers, one ranking, measured

*Part of **15 Days of Modern, Enterprise-Relevant RAG from Scratch**. Lesson: [`docs/day-08/LEARN.md`](../docs/day-08/LEARN.md).*

Day 07 measured two retrievers on 81 labelled queries and found that they fail on **different** queries. Dense retrieval misses 11 exact values ("USD 220", "15 minutes") that BM25 finds at rank 1; BM25 misses a paraphrase ("new dads" for *caregivers*) that dense finds. On the 31 natural questions, 30 are found by at least one of them.

**Hybrid search** runs both retrievers and merges their two rankings into one. This notebook builds that merge from scratch and measures it on Day 07's same queries, users and document snapshot:

1. **Why you can't just add the two scores** (they live on different scales).
2. **Reciprocal Rank Fusion (RRF)**: merge by position, not by score.
3. **Lexical, dense and hybrid, query by query**, against Day 07's accepted baseline, with the paired sign test.
4. **What RRF throws away**, and a **score-based fusion** that keeps it, with its weight chosen by a rule on the dev split and reported on the test split.
5. **Every remaining miss**, with a diagnosis.
6. **Should BM25 and embeddings index the same text?** Day 04's header question, on 81 queries.
7. **Permissions and cost**: what a second retriever adds.

Everything runs locally: no API key, no new packages, about 15 seconds.

## 0. Setup

No new packages. The shared code in [`src/rag_from_scratch/`](../src/rag_from_scratch/) gains **Day 07's evaluation harness** (`evaluation.py`: the labelled query, `validate()`, the metrics, the Wilson interval, the sign test and `compare()`), because today measures with it. `tests/test_shared_code.py` checks that the module returns exactly what the Day 07 notebook returns.

In [1]:
import hashlib
import json
import platform
import statistics
import sys
import time
import warnings
from collections import Counter
from pathlib import Path

assert sys.version_info >= (3, 11), "This series needs Python 3.11 or newer."
# huggingface_hub's progress bars warn when Jupyter widgets aren't installed; they still work as plain text.
warnings.filterwarnings("ignore", message="IProgress not found")

try:
    import numpy as np
    import onnxruntime as ort
except ImportError as error:
    raise ImportError("Day 08 uses Day 04's packages (numpy, onnxruntime, tokenizers, huggingface-hub). From the "
                      "repository root run: python -m pip install -r requirements.txt") from error


def find_repo_root() -> Path:
    # Walk up until the shared package is found, so the notebook works from the repo root or
    # from notebooks/. No hard-coded paths.
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "src" / "rag_from_scratch").is_dir():
            return folder
    raise FileNotFoundError("Could not find src/rag_from_scratch. Start Jupyter from inside the cloned repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))           # import the shared code without installing it

from rag_from_scratch.answering import DIRECTORY, Pipeline              # Day 06's retrieve(), as a user
from rag_from_scratch.bm25 import BM25Index                              # Day 01's BM25
from rag_from_scratch.chunking import section_chunks                     # Day 03's chunker
from rag_from_scratch.embeddings import Embedder                         # Day 04's model
from rag_from_scratch.evaluation import (compare, fingerprint, first_relevant, hit, load_queries,  # Day 07's harness
                                         measure, sign_test, snapshot, tie_margin, top_score,       # (new in src/)
                                         validate, wilson)
from rag_from_scratch.index import VectorIndex, sync                     # Day 05's index
from rag_from_scratch.ingest import ingest                               # Day 02's pipeline

print(f"Python {platform.python_version()} | numpy {np.__version__} | onnxruntime {ort.__version__}")

Python 3.13.17 | numpy 2.4.6 | onnxruntime 1.31.0


In [2]:
TOP_K = 3                # Day 07's definition: an ERROR is an answerable query with no labelled chunk in its top 3
K_VALUES = (1, 3, 5, 10)
RRF_K = 60               # RRF's constant: the original paper's value (Cormack et al., 2009), taken from it, not tuned
EVALS = REPO_ROOT / "evals"
QUESTIONS_FILE = EVALS / "questions-v1.jsonl"
MANIFEST_FILE = EVALS / "manifest.json"
DAY07_BASELINE = EVALS / "baselines" / "day-07-retrieval.json"   # Day 07's accepted run: today's reference
BASELINE_FILE = EVALS / "baselines" / "day-08-retrieval.json"    # today's accepted run, committed and reviewed
OUT_DIR = REPO_ROOT / "outputs" / "day-08"                       # this run's results; outputs/ is git-ignored

## 1. The same queries, the same snapshot, the same two retrievers

A comparison is only fair if everything except the retriever stays the same. So before building anything new, this section loads Day 07's labels with the shared harness, checks that the documents still match the snapshot the labels were written against, rebuilds Day 07's two retrievers exactly, and checks that they **reproduce Day 07's accepted baseline**, query by query.

- **BM25** has no metadata filter, so each user gets an index of the chunks they may read (43 for alice, 44 for maria): filtering before the ranking. It returns only chunks that share at least one word with the query.
- **Dense** is Day 06's `retrieve()`: the permission filter, then a ranking of every chunk the user may read.

In [3]:
manifest = json.loads(MANIFEST_FILE.read_text(encoding="utf-8"))
queries = load_queries(QUESTIONS_FILE)
by_qid = {q.id: q for q in queries}
documents, _ = ingest(REPO_ROOT)
chunks = [c for d in documents for c in section_chunks(d)]
problems = validate(queries, chunks, manifest)
file_sha256 = hashlib.sha256(QUESTIONS_FILE.read_bytes()).hexdigest()
if file_sha256 != manifest["questions_sha256"]:
    problems.insert(0, f"{QUESTIONS_FILE.name} changed but the manifest didn't")
print(f"{len(queries)} labelled queries (version {manifest['version']}) | documents: {len(chunks)} chunks, snapshot "
      f"{fingerprint(snapshot(chunks))[:12]}... | labels written on {manifest['snapshot']['fingerprint'][:12]}...")
print(f"label problems: {len(problems)}" + "".join(f"\n  {p}" for p in problems))
assert not problems, "The labels don't match the documents: run the Day 07 notebook and review them first."

embedder = Embedder(REPO_ROOT)     # Day 04's model: downloaded once (133 MB), SHA-256 checked on every run
index = VectorIndex(embedder.model["repo"], embedder.revision, embedder.dimensions)
sync(index, chunks, embedder.embed)   # the snapshot's chunks, embedded now (Day 05's sync into an empty index)
rag = Pipeline(index, embedder)       # Day 06's retrieve(): as the user, permission filter before the ranking
bm25 = {user: BM25Index({c.chunk_id: c.text for c in chunks if set(c.allowed_groups) & groups})
        for user, groups in DIRECTORY.items()}


def bm25_search(q, k=None):
    # Day 07's BM25: the user's own index. Only chunks that share a word with the query come back.
    return [(h.id, h.score) for h in bm25[q.user].search(q.question, k or len(chunks))]


def dense_search(q, k=None):
    # Day 07's dense retrieval: every chunk the user may read, ranked by cosine similarity
    return [(r.id, r.score) for r in rag.retrieve(q.user, q.question, k or len(chunks))]


# Every query's FULL ranking from each retriever: (chunk_id, score), best first
rankings = {"BM25": {q.id: bm25_search(q) for q in queries}, "dense": {q.id: dense_search(q) for q in queries}}
SELECTIONS = {"questions": [q for q in queries if q.kind == "question" and q.answerable],   # unanswerable ones
              "lookups": [q for q in queries if q.kind == "lookup"]}                        # have no label to find
print(f"BM25: one index per user ({', '.join(f'{u} {len(b.ids)} chunks' for u, b in bm25.items())}) | dense: "
      f"{len(index)} vectors | {len(SELECTIONS['questions'])} answerable questions, {len(SELECTIONS['lookups'])} lookups")

81 labelled queries (version 1) | documents: 44 chunks, snapshot 3f8344d00b8b... | labels written on 3f8344d00b8b...
label problems: 0
tokenizer.json       711,396 bytes   SHA-256 d241a60d5e8f... verified


onnx/model.onnx  133,093,490 bytes   SHA-256 828e1496d7fa... verified


BM25: one index per user (alice 43 chunks, maria 44 chunks) | dense: 44 vectors | 31 answerable questions, 30 lookups


In [4]:
day07 = json.loads(DAY07_BASELINE.read_text(encoding="utf-8"))
DATASET = {"questions": QUESTIONS_FILE.relative_to(REPO_ROOT).as_posix(), "version": manifest["version"],
           "sha256": file_sha256, "snapshot": manifest["snapshot"]["fingerprint"]}
SETTINGS = {   # the two retrievers exactly as Day 07 described them, so compare() checks the settings too
    "BM25": {"code": "rag_from_scratch.bm25 (Day 01)", "k1": bm25["alice"].k1, "b": bm25["alice"].b,
             "indexed_text": "chunk text with its Title > Section header",
             "permissions": "one index of each user's readable chunks (filter before the ranking)"},
    "dense": {"code": "rag_from_scratch.answering.Pipeline.retrieve (Day 06)", "model": embedder.model["repo"],
              "revision": embedder.revision, "query_instruction": True,
              "indexed_text": "chunk text with its Title > Section header",
              "permissions": "metadata filter before the ranking"}}
def entry(name: str, q) -> dict:
    # One query, one retriever, in the baseline's format: the rank of the first labelled chunk, how close that is to a
    # tie (section 10 explains RRF's margin), the top 3 and the best score.
    ranked = rankings[name][q.id]
    margin = rrf_margin(q) if name == "RRF" else tie_margin(ranked, q.answer_chunks)
    return {"first_relevant": first_relevant(ranked, q.answer_chunks) if q.answerable else None,
            "margin": margin if q.answerable else None,
            "top3": [cid for cid, _ in ranked[:TOP_K]], "top_score": top_score(ranked)}


def metrics_of(names) -> dict:
    return {f"{kind}, {name}": {k: round(v, 4) if isinstance(v, float) else v
                                for k, v in measure(rankings[name], selection).items()}
            for kind, selection in SELECTIONS.items() for name in names}


# Day 07's accepted file, with today's labels, settings, ranks and metrics for its two retrievers in place of its own.
# (Its measured thresholds are carried over unchanged: today changes retrieval, not Day 06's thresholds.)
reproduced = {**day07, "dataset": DATASET, "retrievers": {name: SETTINGS[name] for name in ("BM25", "dense")},
              "queries": {q.id: {name: entry(name, q) for name in ("BM25", "dense")} for q in queries},
              "metrics": metrics_of(("BM25", "dense"))}
changes07, ties07 = compare(reproduced, day07)
print(f"BM25 and dense against {DAY07_BASELINE.relative_to(REPO_ROOT).as_posix()}: "
      + (f"{len(changes07)} changes" + "".join(f"\n  {c}" for c in changes07) if changes07 else
         f"no changes ({len(queries)} queries x 2 retrievers: every rank, top 3, best score and metric)"))
for t in ties07:
    print(f"  near tie, not a change: {t}")

BM25 and dense against evals/baselines/day-07-retrieval.json: no changes (81 queries x 2 retrievers: every rank, top 3, best score and metric)


Same labels, same snapshot, same rankings: every difference below comes from the fusion, not from a drift in the inputs.

## 2. Why you can't just add the two scores

The obvious way to combine two retrievers is to add their scores. Here is what the two scales look like, for the question both retrievers missed on Day 07:

In [5]:
WINE, DADS = by_qid["d07-a08"], by_qid["d01-01"]


def show_top(q, names=("BM25", "dense"), n=5) -> None:
    print(f"{q.id}: {q.question} ({q.user}; labelled: {', '.join(q.answer_chunks)})")
    for name in names:
        print(f"  {name}")
        for rank, (cid, score) in enumerate(rankings[name][q.id][:n], start=1):
            print(f"    #{rank} {score:6.3f}  {cid}{'   <- the answer' if cid in q.answer_chunks else ''}")


show_top(WINE)
best = {name: [r[0][1] for r in rankings[name].values() if r] for name in ("BM25", "dense")}
returned = [len(rankings["BM25"][q.id]) for q in queries]
print(f"\nbest score of each query: BM25 {min(best['BM25']):.2f} to {max(best['BM25']):.2f} | dense "
      f"{min(best['dense']):.2f} to {max(best['dense']):.2f}")
print(f"chunks returned per query: BM25 {min(returned)} to {max(returned)} (only chunks sharing a word with the query; "
      f"{sum(n == 0 for n in returned)} queries get none) | dense: every chunk the user may read")


def raw_sum(lists) -> list[tuple[str, float]]:
    # The naive merge: add the scores (a chunk a retriever didn't return adds 0)
    total = {}
    for ranked in lists:
        for cid, score in ranked:
            total[cid] = total.get(cid, 0.0) + score
    return sorted(total.items(), key=lambda kv: (-kv[1], kv[0]))


rankings["raw sum"] = {q.id: raw_sum([rankings["BM25"][q.id], rankings["dense"][q.id]]) for q in queries}
comparable = [q for q in queries if len(rankings["BM25"][q.id]) >= TOP_K]
same_top3 = sum([c for c, _ in rankings["raw sum"][q.id][:TOP_K]] == [c for c, _ in rankings["BM25"][q.id][:TOP_K]]
                for q in comparable)
print(f"raw sum: top 3 identical to BM25's (same chunks, same order) for {same_top3} of the {len(comparable)} queries "
      f"where BM25 returns at least {TOP_K} chunks")

d07-a08: Will Acme pay for wine with a business dinner? (alice; labelled: expense-reimbursement-policy#meals)
  BM25
    #1  6.636  expense-reimbursement-policy#scope
    #2  3.760  travel-policy#purpose
    #3  3.312  parental-leave-policy#returning-to-work
    #4  3.010  data-retention-policy#deletion-requests
    #5  2.776  it-software-requests#approval-times
  dense
    #1  0.686  expense-reimbursement-policy#scope
    #2  0.656  travel-policy#purpose
    #3  0.607  remote-work-policy#purpose
    #4  0.600  expense-reimbursement-policy#meals   <- the answer
    #5  0.555  vpn-access-guide#installing-the-client

best score of each query: BM25 1.71 to 15.28 | dense 0.42 to 0.88
chunks returned per query: BM25 0 to 24 (only chunks sharing a word with the query; 2 queries get none) | dense: every chunk the user may read
raw sum: top 3 identical to BM25's (same chunks, same order) for 62 of the 71 queries where BM25 returns at least 3 chunks


BM25 scores run from about 2 to 15 and depend on the query (how many words match, how rare they are); cosine similarities sit between about 0.4 and 0.9. Added together, the bigger numbers win: **the raw sum is BM25 with dense retrieval as a tie-breaker.** Its top 3 is BM25's top 3, in the same order, for 62 of 71 queries, and it inherits BM25's misses (section 4 measures it).

There are two ways out. Combine **positions** instead of scores (RRF, next), or put the scores on a **common scale** first (section 5).

## 3. Reciprocal Rank Fusion, from scratch

RRF gives every chunk, in every list that returns it, a score of **1 / (k + rank)**, and adds them up:

`RRF(chunk) = Σ over lists of 1 / (k + rank of the chunk in that list)`

Only positions are used, so the two scales never meet. The constant k (60, the original paper's value) flattens the curve: rank 1 is worth 1/61 ≈ 0.0164 and rank 10 is worth 1/70 ≈ 0.0143, so a chunk that **both** lists rank in their top 10 beats a chunk that only one list ranks first. A chunk a list doesn't return gets nothing from that list. Ties are broken by chunk id, like Day 01's BM25, so the order is the same on every machine.

In [6]:
def rrf(lists, k: int = RRF_K) -> list[tuple[str, float]]:
    # Reciprocal Rank Fusion: every list adds 1 / (k + rank) to each chunk it returns. Only positions are used, so the
    # retrievers' scales never meet. Ties (a chunk at ranks (a, b) and another at (b, a)) are broken by chunk id.
    fused = {}
    for ranked in lists:
        for rank, (cid, _) in enumerate(ranked, start=1):
            fused[cid] = fused.get(cid, 0.0) + 1 / (k + rank)
    return sorted(fused.items(), key=lambda kv: (-kv[1], kv[0]))


rankings["RRF"] = {q.id: rrf([rankings["BM25"][q.id], rankings["dense"][q.id]]) for q in queries}


def explain_rrf(q, n: int = 5) -> None:
    ranks = {name: {cid: r for r, (cid, _) in enumerate(rankings[name][q.id], start=1)} for name in ("BM25", "dense")}
    print(f"{q.id}: {q.question}")
    print(f"{'RRF':<5}{'score':>7}  {'chunk':<50}{'BM25':>14}{'dense':>14}")
    for rank, (cid, score) in enumerate(rankings["RRF"][q.id][:n], start=1):
        cells = [f"#{ranks[name][cid]} -> 1/{RRF_K + ranks[name][cid]}" if cid in ranks[name] else "not returned"
                 for name in ("BM25", "dense")]
        print(f"#{rank:<4}{score:>7.4f}  {cid:<50}{cells[0]:>14}{cells[1]:>14}"
              f"{'   <- the answer' if cid in q.answer_chunks else ''}")


explain_rrf(WINE)
tied = [q.id for q in queries if any(a[1] == b[1] for a, b in zip(rankings["RRF"][q.id][:TOP_K + 1],
                                                                      rankings["RRF"][q.id][1:TOP_K + 1]))]
deciding = [q.id for q in queries if q.answerable and len(rankings["RRF"][q.id]) > TOP_K
            and rankings["RRF"][q.id][TOP_K - 1][1] == rankings["RRF"][q.id][TOP_K][1]
            and {rankings["RRF"][q.id][TOP_K - 1][0], rankings["RRF"][q.id][TOP_K][0]} & set(q.answer_chunks)]
print(f"\nexact ties in an RRF top 4: {len(tied)} queries | ties that decide whether a labelled chunk is in the top 3: "
      f"{len(deciding)}")

d07-a08: Will Acme pay for wine with a business dinner?
RRF    score  chunk                                                       BM25         dense
#1    0.0328  expense-reimbursement-policy#scope                    #1 -> 1/61    #1 -> 1/61
#2    0.0323  travel-policy#purpose                                 #2 -> 1/62    #2 -> 1/62
#3    0.0308  expense-reimbursement-policy#meals                    #6 -> 1/66    #4 -> 1/64   <- the answer
#4    0.0299  vpn-access-guide#installing-the-client                #9 -> 1/69    #5 -> 1/65
#5    0.0297  data-retention-policy#deletion-requests               #4 -> 1/64   #11 -> 1/71

exact ties in an RRF top 4: 9 queries | ties that decide whether a labelled chunk is in the top 3: 0


The wine question, which both retrievers missed on Day 07 (BM25 at rank 6, dense at rank 4), is a hit for RRF: no retriever puts the Meals chunk first, but both put it fairly high, and that **agreement** lifts it into the top 3.

Ties are common in RRF (two lists, ranks swapped), so the tie-break rule matters for reproducibility. None of today's ties decides whether a labelled chunk is in the top 3; if one ever did, the result would hang on an arbitrary rule, and the sanity checks would say so.

### 3.1 Does k matter?

k = 60 comes from the paper, not from tuning here (an exploratory run had already swept it while this lesson was being built). For two lists it barely matters: as k grows, RRF orders chunks by the sum of their two ranks; as k shrinks, by their better rank. Between those extremes the order rarely changes. Fusing only each retriever's top 10 (as some search engines do by default) is checked too.

In [7]:
def summary(ranked_by_query, selection) -> str:
    m = measure(ranked_by_query, selection)
    return f"{m['hit@1']:>3}{m['hit@3']:>4}{m['MRR@10']:>6.2f}"


print(f"{'':<26}{'questions (31)':>13}   {'lookups (30)':>12}")
print(f"{'':<26}{'@1  @3   MRR':>13}   {'@1  @3   MRR':>12}")
for k in (1, 5, 20, 60, 100, 1000):
    fused = {q.id: rrf([rankings["BM25"][q.id], rankings["dense"][q.id]], k) for q in queries}
    print(f"{f'RRF, k = {k}':<26}{summary(fused, SELECTIONS['questions'])}   {summary(fused, SELECTIONS['lookups'])}"
          + ("   <- the paper's value, used throughout" if k == RRF_K else ""))
window = {q.id: rrf([rankings["BM25"][q.id][:10], rankings["dense"][q.id][:10]]) for q in queries}
print(f"{'RRF, k = 60, top 10 each':<26}{summary(window, SELECTIONS['questions'])}   "
      f"{summary(window, SELECTIONS['lookups'])}")

                          questions (31)   lookups (30)
                           @1  @3   MRR   @1  @3   MRR
RRF, k = 1                 26  29  0.89    22  30  0.87
RRF, k = 5                 26  30  0.90    21  30  0.84
RRF, k = 20                26  30  0.90    20  30  0.82
RRF, k = 60                26  30  0.90    20  30  0.82   <- the paper's value, used throughout
RRF, k = 100               26  30  0.90    20  30  0.82
RRF, k = 1000              26  30  0.90    20  30  0.82
RRF, k = 60, top 10 each   26  30  0.90    20  30  0.82


From k = 20 to k = 1000 nothing moves. A very small k gives a single first place more weight: one or two more lookups first, and at k = 1 one question drops out of the top 3. Fusing each retriever's top 10 instead of its full ranking gives the same counts as k = 60. **k is not where the gains or the losses come from.** The next section shows where they do.

## 4. Lexical, dense and hybrid, on the same 81 queries

Day 07's metrics for every retriever: hit rate@k, recall@3, MRR@10, and the 95% interval for hit@3. The raw sum is included to show what adding scores does.

In [8]:
def table(names) -> None:
    print(f"{'':<21}{'n':>3}" + "".join(f"{'hit@' + str(k):>7}" for k in K_VALUES) + f"{'recall@3':>10}{'MRR@10':>8}"
          "   hit@3 rate, 95% interval")
    for kind, selection in SELECTIONS.items():
        for name in names:
            m = measure(rankings[name], selection)
            low, high = wilson(m["hit@3"], m["n"])
            print(f"{kind + ', ' + name:<21}{m['n']:>3}" + "".join(f"{m['hit@' + str(k)]:>7}" for k in K_VALUES)
                  + f"{m['recall@3']:>10.2f}{m['MRR@10']:>8.2f}   {m['hit@3'] / m['n']:.0%} ({low:.0%} to {high:.0%})")


table(["BM25", "dense", "raw sum", "RRF"])

                       n  hit@1  hit@3  hit@5 hit@10  recall@3  MRR@10   hit@3 rate, 95% interval
questions, BM25       31     24     29     29     30      0.94    0.85   94% (79% to 98%)
questions, dense      31     26     29     31     31      0.94    0.90   94% (79% to 98%)
questions, raw sum    31     26     29     29     30      0.94    0.88   94% (79% to 98%)
questions, RRF        31     26     30     30     30      0.97    0.90   97% (84% to 99%)
lookups, BM25         30     30     30     30     30      1.00    1.00   100% (89% to 100%)
lookups, dense        30     12     19     26     29      0.62    0.56   63% (46% to 78%)
lookups, raw sum      30     30     30     30     30      1.00    1.00   100% (89% to 100%)
lookups, RRF          30     20     30     30     30      1.00    0.82   100% (89% to 100%)


**In the top 3, RRF is never behind the better retriever**: 30 of 31 questions (each retriever alone: 29) and 30 of 30 lookups, like BM25 (dense alone: 19). The raw sum is BM25 again: the same counts on lookups and the same two misses on questions.

But two numbers go the wrong way. On lookups, BM25 alone puts all 30 answers **first**; RRF puts only 20 first. And a query count of 30 vs 29 means nothing until we know which queries moved. So, as on Day 07, compare **pair by pair**, on the same queries, with the sign test (if neither retriever were better, each disagreement would be a coin flip):

In [9]:
PAIRED = {}


def show_paired(a: str, b: str, k: int = TOP_K, kinds=("questions", "lookups")) -> None:
    for kind in kinds:
        selection = SELECTIONS[kind]
        found = {name: {q.id for q in selection if hit(rankings[name][q.id], q.answer_chunks, k)} for name in (a, b)}
        groups = {"both": found[a] & found[b], f"only {a}": found[a] - found[b], f"only {b}": found[b] - found[a],
                  "neither": {q.id for q in selection} - found[a] - found[b]}
        PAIRED[a, b, k, kind] = {label: sorted(ids) for label, ids in groups.items()}
        p = sign_test(len(groups[f"only {a}"]), len(groups[f"only {b}"]))
        where = "first" if k == 1 else f"in the top {k}"
        print(f"{a} vs {b}, {kind} ({len(selection)}), labelled chunk {where}: "
              + " | ".join(f"{label} {len(ids)}" for label, ids in groups.items()) + f" | sign test p = {p:.2g}")
        for label in (f"only {a}", f"only {b}", "neither"):
            ids = sorted(groups[label])
            if ids and len(ids) <= 3:
                print(f"    {label:<14} " + ", ".join(f"{qid} ({by_qid[qid].question})" for qid in ids))
            elif ids:
                print(f"    {label:<14} " + ", ".join(ids))


show_paired("RRF", "BM25")
show_paired("RRF", "dense")
print()
show_paired("RRF", "BM25", k=1, kinds=("lookups",))

RRF vs BM25, questions (31), labelled chunk in the top 3: both 29 | only RRF 1 | only BM25 0 | neither 1 | sign test p = 1
    only RRF       d07-a08 (Will Acme pay for wine with a business dinner?)
    neither        d01-01 (How much time off do new dads get?)
RRF vs BM25, lookups (30), labelled chunk in the top 3: both 30 | only RRF 0 | only BM25 0 | neither 0 | sign test p = 1
RRF vs dense, questions (31), labelled chunk in the top 3: both 28 | only RRF 2 | only dense 1 | neither 0 | sign test p = 1
    only RRF       d03-13 (How soon must I report a lost laptop?), d07-a08 (Will Acme pay for wine with a business dinner?)
    only dense     d01-01 (How much time off do new dads get?)
RRF vs dense, lookups (30), labelled chunk in the top 3: both 19 | only RRF 11 | only dense 0 | neither 0 | sign test p = 0.00098
    only RRF       d04-v06, d04-v07, d04-v10, d04-v11, d04-v12, d04-v14, d04-v17, d04-v18, d04-v20, d04-v21, d04-v24

RRF vs BM25, lookups (30), labelled chunk first: both 20 

Read the counts:

- **Against dense, on lookups: a clear win.** RRF finds 11 answers that dense misses and loses none (p ≈ 0.001). That is Day 07's biggest error category, fixed.
- **On questions: one each way, or close to it.** RRF gains the wine question over BM25; over dense it gains the lost laptop and wine but **loses new dads**. With so few disagreements the sign test can't tell them apart (p = 1).
- **Against BM25, on lookups at rank 1: a clear loss.** BM25 puts 10 answers first that RRF doesn't (p ≈ 0.002). In the top 3 they are level (30 and 30), so a reader of three chunks still sees them; a user who looks at the first result doesn't.

So RRF is not "both retrievers' strengths". It has two losses, and both come from what it throws away.

### 4.1 What RRF throws away

In [10]:
explain_rrf(DADS, n=3)
bm25_hits = [cid for cid, _ in rankings["BM25"][DADS.id]]
dense_rank = {cid: r for r, (cid, _) in enumerate(rankings["dense"][DADS.id], start=1)}
rrf_rank = {cid: r for r, (cid, _) in enumerate(rankings["RRF"][DADS.id], start=1)}
print(f"   ... the {len(bm25_hits)} chunks BM25 returns fill RRF's ranks 1 to {max(rrf_rank[c] for c in bm25_hits)}, "
      f"including BM25's #1, which dense ranks #{dense_rank[bm25_hits[0]]}. Then dense's top 3:")
for cid, _ in rankings["dense"][DADS.id][:3]:
    cell = f"#{dense_rank[cid]} -> 1/{RRF_K + dense_rank[cid]}"
    print(f"#{rrf_rank[cid]:<4}{dict(rankings['RRF'][DADS.id])[cid]:>7.4f}  {cid:<50}{'not returned':>14}{cell:>14}"
          f"{'   <- the answer' if cid in DADS.answer_chunks else ''}")
print()

LOCKOUT = by_qid["d04-v07"]          # the lookup "15 minutes": dense's worst value miss on Day 07
show_top(LOCKOUT, names=("BM25",), n=3)
print(f"  dense ranks the answer #{first_relevant(rankings['dense'][LOCKOUT.id], LOCKOUT.answer_chunks)}, "
      f"RRF #{first_relevant(rankings['RRF'][LOCKOUT.id], LOCKOUT.answer_chunks)}")
demoted = [q.id for q in SELECTIONS["lookups"] if first_relevant(rankings["BM25"][q.id], q.answer_chunks) == 1
           and first_relevant(rankings["RRF"][q.id], q.answer_chunks) != 1]
print(f"\nlookups BM25 ranks first and RRF doesn't: {len(demoted)}")

d01-01: How much time off do new dads get?
RRF    score  chunk                                                       BM25         dense
#1    0.0303  remote-work-policy#core-hours                         #6 -> 1/66    #6 -> 1/66
#2    0.0299  new-starter-it-checklist#training                     #4 -> 1/64   #10 -> 1/70
#3    0.0291  remote-work-policy#purpose                            #2 -> 1/62   #17 -> 1/77
   ... the 12 chunks BM25 returns fill RRF's ranks 1 to 12, including BM25's #1, which dense ranks #39. Then dense's top 3:
#13   0.0164  parental-leave-policy#returning-to-work             not returned    #1 -> 1/61
#14   0.0161  parental-leave-policy#who-is-eligible               not returned    #2 -> 1/62
#15   0.0159  parental-leave-policy#leave-entitlement             not returned    #3 -> 1/63   <- the answer

d04-v07: 15 minutes (maria; labelled: password-and-mfa-standard#account-lockout)
  BM25
    #1  6.283  password-and-mfa-standard#account-lockout   <- the answer
    

**Agreement wins, even over a first place.** For "new dads", BM25 returns 12 chunks (anything with *time*, *off* or *new*), none of them about parental leave. Each of those 12 gets two votes, one from each list, and two small votes beat one large one: they fill RRF's ranks 1 to 12, including BM25's first, which dense ranks 39th. Dense's own top 3, the three parental-leave chunks, get one vote each and come 13th, 14th and 15th; the answer is 15th. RRF assumes that a chunk only one retriever finds is less likely to be relevant. On a vocabulary mismatch, where only one retriever *can* find it, that assumption is exactly wrong.

**Confidence is ignored.** For "15 minutes", BM25's first chunk scores more than twice its second (6.28 vs 3.03); dense ranks the answer 19th. RRF sees only the two positions, 1 and 19, and ranks the answer 2nd. The size of BM25's lead, the strongest evidence in the query, never reaches the fusion. The same happens to 10 of the 30 lookups.

Both losses come from discarding the scores. That suggests the alternative.

## 5. A score-based fusion: does it earn its place?

Put each retriever's scores on the same 0-to-1 scale, then take a weighted average:

`fused(chunk) = α × dense_normalised + (1 − α) × BM25_normalised`

**Min-max normalisation** maps a retriever's best score for this query to 1 and its worst to 0; a chunk BM25 didn't return gets 0 from BM25. This keeps the *shape* of each ranking: BM25's big lead on "15 minutes" survives as a big gap. It costs something RRF doesn't need: **a weight α, which has to be chosen on labelled data**.

The rule: *the α, in steps of 0.1, that puts a labelled chunk in the top 3 for the most **dev** queries (16 answerable dev questions and the 30 lookups); ties go to the higher dev MRR@10, then to the α closest to 0.5.* It uses dev queries only, and the sweep below scores nothing else. **It wasn't pre-registered**, though: while this lesson was being built, exploratory runs had already printed results for several weights on both splits, test included, before the rule was written here. (Day 07's threshold rule had the same flaw; its LEARN.md, section 9.2, has the correction.) So read the test-split numbers below as a check, not as clean held-out evidence.

In [11]:
def min_max(ranked) -> dict[str, float]:
    # One retriever's scores for one query, rescaled to 0..1: its best chunk 1, its worst 0
    if not ranked:
        return {}
    hi, lo = max(s for _, s in ranked), min(s for _, s in ranked)
    return {cid: (s - lo) / (hi - lo) if hi > lo else 1.0 for cid, s in ranked}


def weighted(lexical, dense, alpha: float) -> list[tuple[str, float]]:
    # alpha * dense + (1 - alpha) * BM25, both rescaled to 0..1 first. A chunk BM25 didn't return gets 0 from BM25.
    lex, den = min_max(lexical), min_max(dense)
    fused = {cid: alpha * den.get(cid, 0.0) + (1 - alpha) * lex.get(cid, 0.0) for cid in {*lex, *den}}
    return sorted(fused.items(), key=lambda kv: (-kv[1], kv[0]))


DEV = [q for q in queries if q.split == "dev" and q.answerable]          # the queries we may tune on
DEV_PARTS = {"dev questions": [q for q in DEV if q.kind == "question"], "lookups": [q for q in DEV if q.kind == "lookup"],
             "all dev": DEV}
ALPHAS = [i / 10 for i in range(11)]
sweep = {}
for alpha in ALPHAS:
    fused = {q.id: weighted(rankings["BM25"][q.id], rankings["dense"][q.id], alpha) for q in DEV}
    sweep[alpha] = {part: measure(fused, selection) for part, selection in DEV_PARTS.items()}

# The rule (dev only; not pre-registered, see above): most dev queries in the top 3, then higher dev MRR, then closest to 0.5
ALPHA = max(ALPHAS, key=lambda a: (sweep[a]["all dev"]["hit@3"], sweep[a]["all dev"]["MRR@10"], -abs(a - 0.5)))

print(f"{'alpha':<7}{'dev questions (16)':>18}   {'lookups (30)':>13}   {'all dev (46)':>13}")
print(f"{'':<7}{'@1  @3   MRR':>18}   {'@1  @3   MRR':>13}   {'@3   MRR':>13}")
for alpha in ALPHAS:
    m = sweep[alpha]
    label = {0.0: "  (BM25 alone)", 1.0: "  (dense alone)"}.get(alpha, "")
    print(f"{alpha:<7.1f}" + "".join(f"{m[p]['hit@1']:>9}{m[p]['hit@3']:>4}{m[p]['MRR@10']:>6.2f}   "
                                      for p in ("dev questions", "lookups"))
          + f"{m['all dev']['hit@3']:>7}{m['all dev']['MRR@10']:>6.2f}" + ("  <- the rule's choice" if alpha == ALPHA else label))
rankings["weighted"] = {q.id: weighted(rankings["BM25"][q.id], rankings["dense"][q.id], ALPHA) for q in queries}

alpha  dev questions (16)    lookups (30)    all dev (46)
             @1  @3   MRR    @1  @3   MRR        @3   MRR
0.0           13  15  0.85          30  30  1.00        45  0.95  (BM25 alone)
0.1           13  15  0.85          30  30  1.00        45  0.95
0.2           13  15  0.85          30  30  1.00        45  0.95
0.3           13  15  0.85          30  30  1.00        45  0.95
0.4           13  15  0.86          30  30  1.00        45  0.95
0.5           13  15  0.87          30  30  1.00        45  0.96
0.6           13  15  0.89          30  30  1.00        45  0.96
0.7           13  16  0.89          30  30  1.00        46  0.96  <- the rule's choice
0.8           13  16  0.89          26  29  0.91        45  0.90
0.9           13  15  0.88          18  25  0.72        40  0.77
1.0           13  15  0.88          12  19  0.56        34  0.67  (dense alone)


Two regions show up. For every α up to 0.7, **all 30 lookups stay at rank 1**: BM25's lead on an exact value is large enough to survive the averaging. From α = 0.7 up, **all 16 dev questions** are in the top 3. Only α = 0.7 has both, so the rule picks it. (Weaviate's hybrid search, which also defaults to a min-max fusion, uses α = 0.75 when none is given; that is a coincidence of the same shape, not evidence.)

The lookup result doesn't depend on tuning: it holds over a wide range of weights. The question result does: it depends on getting α roughly right, on 16 dev questions. Now the test split, which the rule doesn't use:

### 5.1 All of them, on both splits

In [12]:
NAMES = ["BM25", "dense", "RRF", "weighted"]
table(NAMES)
print()
for split in ("dev", "test"):
    part = [q for q in SELECTIONS["questions"] if q.split == split]
    print(f"answerable questions, {split} ({len(part)}): " + " | ".join(
        f"{name} {measure(rankings[name], part)['hit@1']}/{measure(rankings[name], part)['hit@3']}" for name in NAMES)
          + "   (first / in the top 3)")
print()
show_paired("weighted", "RRF")
show_paired("weighted", "RRF", k=1)
show_paired("weighted", "BM25", k=1, kinds=("lookups",))
show_paired("weighted", "dense", kinds=("questions",))

                       n  hit@1  hit@3  hit@5 hit@10  recall@3  MRR@10   hit@3 rate, 95% interval
questions, BM25       31     24     29     29     30      0.94    0.85   94% (79% to 98%)
questions, dense      31     26     29     31     31      0.94    0.90   94% (79% to 98%)
questions, RRF        31     26     30     30     30      0.97    0.90   97% (84% to 99%)
questions, weighted   31     27     31     31     31      1.00    0.92   100% (89% to 100%)
lookups, BM25         30     30     30     30     30      1.00    1.00   100% (89% to 100%)
lookups, dense        30     12     19     26     29      0.62    0.56   63% (46% to 78%)
lookups, RRF          30     20     30     30     30      1.00    0.82   100% (89% to 100%)
lookups, weighted     30     30     30     30     30      1.00    1.00   100% (89% to 100%)

answerable questions, dev (16): BM25 13/15 | dense 13/15 | RRF 13/15 | weighted 13/16   (first / in the top 3)
answerable questions, test (15): BM25 11/14 | dense 13/14 | RR

**The weighted fusion has no miss in the top 3**: 31 of 31 questions, 30 of 30 lookups, and every lookup first, like BM25 alone. Against RRF it gains new dads in the top 3 and 10 lookups at rank 1 (p ≈ 0.002 for those); against dense alone it gains the lost laptop and wine (p = 0.5).

How much of that is earned? Be precise about which numbers are clean:

- **The lookups are all dev queries**, and so are 16 of the questions. α was chosen on them, so those counts are optimistic by construction. The lookup advantage, though, holds for every α from 0 to 0.7, so it is a property of score fusion, not of the tuning.
- **On the 15 test questions**, which the rule doesn't use (but whose results had been seen while exploring), the weighted fusion and RRF both have 15 of 15 in the top 3, and 14 vs 13 first. On test alone the two can't be told apart, and even this tie is a check, not a clean held-out result.

**Decision for this corpus:** the weighted fusion (α = 0.7) becomes the hybrid that later days build on, because it avoids both of RRF's measured losses and loses nothing measured. RRF stays in the baseline as the **parameter-free** alternative: it needs no labels, no normalisation and no re-tuning when the documents change, which is why search engines offer it for hybrid queries. If you have no labelled queries, start with RRF.

## 6. Every remaining miss, with a reason

Day 07's 15 misses, traced through both fusions, and the diagnosis of every miss that is left. As on Day 07, the `assert` fails if the set of misses changes, so a new miss can't go unexplained.

In [13]:
DIAGNOSES = {(e["retriever"], e["id"]): (e["category"], e["why"]) for e in day07["errors"]}   # Day 07's, unchanged
DIAGNOSES[("RRF", "d01-01")] = ("one retriever only", "BM25 never returns it: one vote (dense's #3) against two "
                                                      "for each of the 12 chunks BM25 does return")

misses = [(name, q) for name in NAMES for q in queries if q.answerable and not hit(rankings[name][q.id], q.answer_chunks, TOP_K)]
assert {(name, q.id) for name, q in misses} == set(DIAGNOSES), "every miss needs a diagnosis, and only misses"

print(f"Day 07's misses ({sum(name in ('BM25', 'dense') for name, _ in misses)}), and the same queries after fusion "
      "(rank of the first labelled chunk; - = never returned):\n")
print(f"{'category':<15}{'query':<9}{'missed by':<11}{'BM25':>6}{'dense':>7}{'RRF':>6}{'weighted':>10}   question or lookup")
fmt = lambda r: "-" if r is None else str(r)
for name, q in sorted(((n, q) for n, q in misses if n in ("BM25", "dense")), key=lambda m: (DIAGNOSES[m[0], m[1].id][0], m[1].id)):
    r = [first_relevant(rankings[n][q.id], q.answer_chunks) for n in NAMES]
    print(f"{DIAGNOSES[name, q.id][0]:<15}{q.id:<9}{name:<11}{fmt(r[0]):>6}{fmt(r[1]):>7}{fmt(r[2]):>6}{fmt(r[3]):>10}   "
          f"{q.question}")
print("\nmisses after fusion:")
for name, q in misses:
    if name not in ("BM25", "dense"):
        category, why = DIAGNOSES[name, q.id]
        print(f"  {name:<9}{q.id:<9}{category:<20}{q.question}\n{'':<38}{why}")
print(f"  weighted: {sum(name == 'weighted' for name, _ in misses)} misses")

Day 07's misses (15), and the same queries after fusion (rank of the first labelled chunk; - = never returned):

category       query    missed by    BM25  dense   RRF  weighted   question or lookup
broader chunk  d07-a08  dense           6      4     3         3   Will Acme pay for wine with a business dinner?
exact value    d04-v06  dense           1      4     1         1   14 days
exact value    d04-v07  dense           1     19     2         1   15 minutes
exact value    d04-v10  dense           1      7     2         1   3 business days
exact value    d04-v11  dense           1      6     2         1   30 days
exact value    d04-v12  dense           1      4     2         1   5 business days
exact value    d04-v14  dense           1      9     1         1   60 days
exact value    d04-v17  dense           1      4     2         1   USD 180
exact value    d04-v18  dense           1      4     2         1   USD 220
exact value    d04-v20  dense           1      4     3         1   U

Every one of Day 07's exact-value misses is fixed by both fusions, and so are the lost laptop (*other section*) and wine (*vocabulary* for BM25, *broader chunk* for dense). The one miss left is RRF's on new dads, diagnosed above: a chunk found by only one retriever. Wine only just makes it: rank 3 in both fusions, with the expense policy's Scope still first.

"No miss in the top 3" is not "no problem left". The ranking within the top 3 still matters (section 5.1 counts first places), and five unanswerable test questions still get a confident, wrong answer from Day 06's pipeline (Day 07, section 5.3): retrieval can't fix that. That is Day 09's starting point.

## 7. Should BM25 and embeddings index the same text?

Every chunk's text starts with a `Title > Section` header (Day 03). Day 04 found, on 15 questions, that the header **helps BM25 and hurts embeddings**. With a hybrid, each retriever could index its own version of the text. Before building that, measure it on 81 queries: both retrievers with and without the header, alone and fused (α stays at 0.7, not re-tuned).

In [14]:
def body(c) -> str:
    # A chunk's text without its first line, the "Title > Section" header (Day 03)
    header, text = c.text.split("\n", 1)
    assert header == f"{c.title} > {c.section}"
    return text


plain_bm25 = {user: BM25Index({c.chunk_id: body(c) for c in chunks if set(c.allowed_groups) & groups})
              for user, groups in DIRECTORY.items()}
plain_index = VectorIndex(embedder.model["repo"], embedder.revision, embedder.dimensions)
plain_index.upsert(chunks, embedder.embed([body(c) for c in chunks]))   # same records, vectors of the text alone
texts = {"BM25": {"header": rankings["BM25"],
                  "no header": {q.id: [(h.id, h.score) for h in plain_bm25[q.user].search(q.question, len(chunks))]
                                for q in queries}},
         "dense": {"header": rankings["dense"],
                   "no header": {q.id: [(r.id, r.score) for r in rag.retrieve(q.user, q.question, len(chunks),
                                                                              from_index=plain_index)]
                                 for q in queries}}}

print(f"{'':<42}{'questions (31)':>13}   {'lookups (30)':>12}")
print(f"{'':<42}{'@1  @3   MRR':>13}   {'@1  @3   MRR':>12}")
for name in ("BM25", "dense"):
    for version in ("header", "no header"):
        r = texts[name][version]
        print(f"{name + ', ' + version:<42}{summary(r, SELECTIONS['questions'])}   {summary(r, SELECTIONS['lookups'])}")
for fusion in ("RRF", "weighted"):
    for b_version in ("header", "no header"):
        for d_version in ("header", "no header"):
            lists = lambda q: (texts["BM25"][b_version][q.id], texts["dense"][d_version][q.id])
            fused = {q.id: rrf(lists(q)) if fusion == "RRF" else weighted(*lists(q), ALPHA) for q in queries}
            label = f"{fusion}: BM25 {b_version}, dense {d_version}"
            print(f"{label:<42}{summary(fused, SELECTIONS['questions'])}   {summary(fused, SELECTIONS['lookups'])}"
                  + ("   <- today" if b_version == d_version == "header" else ""))
print()
for name in ("BM25", "dense"):
    for k in (1, TOP_K):
        for kind, selection in SELECTIONS.items():
            with_h = {q.id for q in selection if hit(texts[name]["header"][q.id], q.answer_chunks, k)}
            without = {q.id for q in selection if hit(texts[name]["no header"][q.id], q.answer_chunks, k)}
            gain, loss = sorted(without - with_h), sorted(with_h - without)
            if gain or loss:
                print(f"{name} without the header, {kind}, {'first' if k == 1 else 'top 3'}: gains {len(gain)} "
                      f"{gain}, loses {len(loss)} {loss}, sign test p = {sign_test(len(gain), len(loss)):.2g}")

                                          questions (31)   lookups (30)
                                           @1  @3   MRR   @1  @3   MRR
BM25, header                               24  29  0.85    30  30  1.00
BM25, no header                            22  28  0.81    30  30  1.00
dense, header                              26  29  0.90    12  19  0.56
dense, no header                           27  31  0.92    12  19  0.57
RRF: BM25 header, dense header             26  30  0.90    20  30  0.82   <- today
RRF: BM25 header, dense no header          26  30  0.90    19  29  0.79
RRF: BM25 no header, dense header          25  29  0.87    20  30  0.82
RRF: BM25 no header, dense no header       25  29  0.88    19  29  0.79
weighted: BM25 header, dense header        27  31  0.92    30  30  1.00   <- today
weighted: BM25 header, dense no header     28  31  0.95    30  30  1.00
weighted: BM25 no header, dense header     26  31  0.90    30  30  1.00
weighted: BM25 no header, dense no header  

**Alone, Day 04's direction holds on 81 queries:** without the header, dense retrieval finds all 31 questions in the top 3 (29 with it), and BM25 puts fewer answers first (22 vs 24) and loses one from its top 3. **None of these differences is significant**: two queries one way and none the other is p = 0.5.

**Fused, the choice hardly matters:** no combination puts more answers in the top 3 than today's. The best alternative, the weighted fusion with headerless vectors, puts one more question first (28 vs 27); the same vectors cost RRF a lookup. A separate text per retriever means a second embedding pass and a second thing to keep in sync, for no measured gain. **Decision: keep one text for both retrievers.** If the labelled set grows and the dense advantage holds, revisit it with the new numbers.

## 8. Permissions: fuse lists that were each filtered

A hybrid has two retrievers, so it has **two places where the permission filter must be applied**. Both of today's inputs are filtered before ranking, so nothing a user may not read can enter a fused list: checked here for every query and every retriever, in the top 10.

What if one of them weren't? Day 07's BM25 builds one index per user; suppose it used a single shared index instead.

In [15]:
leaks = [(name, q.id) for name in rankings for q in queries
         if set(q.restricted_chunks) & {cid for cid, _ in rankings[name][q.id][:10]}]
print(f"restricted chunks in any top 10 (all {len(rankings)} retrievers, {len(queries)} queries): {len(leaks)}")

shared_bm25 = BM25Index({c.chunk_id: c.text for c in chunks})        # the mistake: one index, no permission filter
readable = {u: {c.chunk_id for c in chunks if set(c.allowed_groups) & g} for u, g in DIRECTORY.items()}
print("\nwith a shared, unfiltered BM25 index (dense still filtered):")
EXPOSED = {}
for name in ("RRF", "weighted"):
    exposed = []
    for q in queries:
        leaky = [(h.id, h.score) for h in shared_bm25.search(q.question, len(chunks))]
        fused = rrf([leaky, rankings["dense"][q.id]]) if name == "RRF" else weighted(leaky, rankings["dense"][q.id], ALPHA)
        bad = [(rank, cid) for rank, (cid, _) in enumerate(fused, start=1) if cid not in readable[q.user]]
        if bad:
            exposed.append((q, bad[0]))
    EXPOSED[name] = exposed
    print(f"  {name}: {len(exposed)} queries ({', '.join(sorted({q.user for q, _ in exposed}))}) get a chunk the asker may "
          f"not read into the fused ranking; best rank {min(rank for _, (rank, _) in exposed)}, in the top 10 for "
          f"{sum(rank <= 10 for _, (rank, _) in exposed)}, in the top 3 for {sum(rank <= 3 for _, (rank, _) in exposed)}")

restricted chunks in any top 10 (all 5 retrievers, 81 queries): 0

with a shared, unfiltered BM25 index (dense still filtered):
  RRF: 8 queries (alice) get a chunk the asker may not read into the fused ranking; best rank 4, in the top 10 for 4, in the top 3 for 0
  weighted: 8 queries (alice) get a chunk the asker may not read into the fused ranking; best rank 22, in the top 10 for 0, in the top 3 for 0


With a shared BM25 index, 8 of alice's queries get the managers-only guide **into their fused ranking**. With RRF it reaches her top 10 for 4 of them, once at rank 4, one place below what Day 06's reader reads; with the weighted fusion it stays below rank 20. It misses her top 3 only because the filtered dense list never votes for it: luck of the fusion, not a defence. A deeper context, a longer result page or another weight would show it. **A permission leak that happens to rank low is still a leak.** Apply the filter in every retriever, before fusion, and keep Day 06's re-check after it (it raises an error before anything is sent).

## 9. What the second retriever costs

Hybrid search runs both retrievers on every query, so its latency is BM25 plus dense plus the fusion (or the slower of the two, if they run in parallel). Measured here, on this machine, over the 81 queries (3 rounds; not asserted, except that fusion is cheap):

In [16]:
def timed(work, rounds: int = 3) -> tuple[float, float]:
    samples = []
    for _ in range(rounds):
        for q in queries:
            started = time.perf_counter()
            work(q)
            samples.append((time.perf_counter() - started) * 1000)
    samples.sort()
    return statistics.median(samples), samples[int(0.95 * len(samples)) - 1]


timings = {
    "BM25 (full ranking)": timed(bm25_search),
    "dense (embed the question + search)": timed(dense_search),
    "RRF fusion": timed(lambda q: rrf([rankings["BM25"][q.id], rankings["dense"][q.id]])),
    "weighted fusion": timed(lambda q: weighted(rankings["BM25"][q.id], rankings["dense"][q.id], ALPHA)),
}
for step, (median, p95) in timings.items():
    print(f"{step:<38} median {median:7.3f} ms   p95 {p95:7.3f} ms")

BM25 (full ranking)                    median   0.016 ms   p95   0.055 ms
dense (embed the question + search)    median   9.628 ms   p95  12.694 ms
RRF fusion                             median   0.013 ms   p95   0.022 ms
weighted fusion                        median   0.027 ms   p95   0.045 ms


Embedding the question dominates; BM25 and either fusion add a fraction of a millisecond on 44 chunks. At a larger scale BM25 needs an inverted index on disk and the fusion needs each retriever's top N rather than a full ranking, but the shape holds: **the second retriever is cheap next to the model call, and the fusion is cheaper still.** Day 13 measures the whole pipeline.

## 10. Save the run, compare with the accepted baseline

The run is written to `outputs/day-08/retrieval-baseline.json` in Day 07's format, with four retrievers: BM25 and dense (unchanged), RRF and the weighted fusion. The accepted copy is committed as [`evals/baselines/day-08-retrieval.json`](../evals/baselines/day-08-retrieval.json); Days 09–11 compare against it.

One detail differs from Day 07: how close a ranking is to a tie. RRF scores are made of ranks, so CPU rounding can move an RRF rank only by swapping two near-equal scores **in one of its inputs**. Its margin is therefore the smallest input gap whose swap would move the labelled chunk. The weighted fusion's margin is its own score gap, like BM25's and dense's.

In [17]:
def rrf_margin(q) -> float | None:
    # The smallest score gap in an INPUT ranking (BM25 or dense) whose swap would move the labelled chunk's RRF rank
    if not q.answerable:
        return None
    lists = [rankings["BM25"][q.id], rankings["dense"][q.id]]
    base, gaps = first_relevant(rankings["RRF"][q.id], q.answer_chunks), []
    for i, inputs in enumerate(lists):
        for j in range(len(inputs) - 1):
            swapped = inputs[:j] + [inputs[j + 1], inputs[j]] + inputs[j + 2:]
            if first_relevant(rrf(lists[:i] + [swapped] + lists[i + 1:]), q.answer_chunks) != base:
                gaps.append(inputs[j][1] - inputs[j + 1][1])
    return round(min(gaps), 4) if gaps else None


SETTINGS.update({
    "RRF": {"code": "rrf() in the Day 08 notebook", "fuses": ["BM25", "dense"], "k": RRF_K,
            "inputs": "full rankings, each already permission-filtered", "ties": "chunk id"},
    "weighted": {"code": "weighted() in the Day 08 notebook", "fuses": ["BM25", "dense"], "alpha_dense": ALPHA,
                 "normalisation": "min-max per retriever and query; 0 from a retriever that didn't return the chunk",
                 "alpha_rule": "most dev queries in the top 3, then the higher dev MRR@10, then closest to 0.5",
                 "inputs": "full rankings, each already permission-filtered", "ties": "chunk id"}})

baseline = {
    "name": "Day 08 retrieval baseline",
    "dataset": DATASET,
    "retrievers": {name: SETTINGS[name] for name in NAMES},
    "top_k": TOP_K,
    "metrics": metrics_of(NAMES),
    "queries": {q.id: {name: entry(name, q) for name in NAMES} for q in queries},
    "errors": [{"retriever": name, "id": q.id, "first_relevant": first_relevant(rankings[name][q.id], q.answer_chunks),
                "category": DIAGNOSES[name, q.id][0], "why": DIAGNOSES[name, q.id][1]} for name, q in misses],
    "thresholds": "not re-measured on Day 08 (retrieval only): see evals/baselines/day-07-retrieval.json",
}
OUT_DIR.mkdir(parents=True, exist_ok=True)
run_file = OUT_DIR / "retrieval-baseline.json"
run_file.write_text(json.dumps(baseline, indent=1, ensure_ascii=False) + "\n", encoding="utf-8", newline="\n")
print(f"this run: {run_file.relative_to(REPO_ROOT).as_posix()} ({len(baseline['queries'])} queries x {len(NAMES)} "
      f"retrievers, {len(baseline['errors'])} errors)")
if BASELINE_FILE.is_file():
    changes, ties = compare(baseline, json.loads(BASELINE_FILE.read_text(encoding="utf-8")))
    print(f"compared with {BASELINE_FILE.relative_to(REPO_ROOT).as_posix()}: "
          + (f"{len(changes)} changes" + "".join(f"\n  {c}" for c in changes) if changes else
             f"no changes ({len(queries)} queries x {len(NAMES)} retrievers and their metrics)"))
    for t in ties:
        print(f"  near tie, not a change: {t}")
else:
    changes, ties = None, []
    print(f"no accepted baseline yet: review {run_file.name}, then copy it to "
          f"{BASELINE_FILE.relative_to(REPO_ROOT).as_posix()} and commit it")
fragile = sorted((entry_["margin"], qid, name) for qid, per in baseline["queries"].items()
                 for name, entry_ in per.items() if entry_["margin"] is not None and entry_["margin"] < 0.002)
print("closest calls in this run: " + ", ".join(f"{qid} {name} ({m})" for m, qid, name in fragile))

this run: outputs/day-08/retrieval-baseline.json (81 queries x 4 retrievers, 16 errors)
compared with evals/baselines/day-08-retrieval.json: no changes (81 queries x 4 retrievers and their metrics)
closest calls in this run: d03-13 RRF (0.0001), d03-13 dense (0.0001), d04-i03 dense (0.0008), d04-v10 RRF (0.0009), d04-v10 dense (0.0009), d07-a03 weighted (0.0016), d04-v21 RRF (0.0017), d04-v21 dense (0.0017)


The closest calls are the ones Day 07 already listed for dense, and they carry into RRF: if another CPU swapped dense's two near-tied chunks for the lost-laptop question (0.0001 apart), RRF's rank for it would move from 2 to 3; for "3 business days" (0.0009), from 2 to 1. The comparison reports such a move as a **near tie**, not a change. The weighted fusion's closest call is 0.0016.

## 11. Sanity checks

Each check pins a result quoted in this notebook or in LEARN.md. Scores are compared with a tolerance, ids, ranks and counts exactly.

In [18]:
def close(x: float, expected: float, tol: float = 0.01) -> bool:
    return abs(x - expected) <= tol


def ranks(name: str, ids) -> list:
    return [first_relevant(rankings[name][qid], by_qid[qid].answer_chunks) for qid in ids]


def counts(name: str, kind: str) -> tuple:
    m = measure(rankings[name], SELECTIONS[kind])
    return m["hit@1"], m["hit@3"], m["hit@5"], m["hit@10"]


# The same inputs as Day 07 (section 1): compare() checks ranks, scores and metrics; the top 3 lists are checked here
assert problems == [] and changes07 == [] and len(queries) == 81 and len(chunks) == 44
assert all(reproduced["queries"][qid][n]["top3"] == day07["queries"][qid][n]["top3"] for qid in by_qid for n in ("BM25", "dense"))
assert {u: len(b.ids) for u, b in bm25.items()} == {"alice": 43, "maria": 44} and len(index) == 44
assert counts("BM25", "questions") == (24, 29, 29, 30) and counts("dense", "lookups") == (12, 19, 26, 29)

# Score mismatch (section 2)
assert 1.5 < min(best["BM25"]) < 2.0 and 15.0 < max(best["BM25"]) < 15.5 and close(min(best["dense"]), 0.42)
assert same_top3 == 62 and len(comparable) == 71 and sum(n == 0 for n in returned) == 2
assert ranks("raw sum", ["d01-01", "d07-a08"]) == [15, 6] and counts("raw sum", "lookups")[0] == 30

# RRF (sections 3 and 4)
assert [cid for cid, _ in rankings["RRF"][WINE.id][:3]][2] == WINE.answer_chunks[0] and ranks("RRF", [WINE.id]) == [3]
assert dict(rankings["RRF"][WINE.id])[WINE.answer_chunks[0]] == 1 / (RRF_K + 6) + 1 / (RRF_K + 4)   # BM25 #6, dense #4
assert len(tied) == 9 and deciding == []
assert counts("RRF", "questions") == (26, 30, 30, 30) and counts("RRF", "lookups") == (20, 30, 30, 30)
assert close(measure(rankings["RRF"], SELECTIONS["lookups"])["MRR@10"], 0.822, 0.001)
assert PAIRED["RRF", "dense", 3, "lookups"]["only RRF"] == sorted(e["id"] for e in day07["errors"]
                                                                 if e["category"] == "exact value")
assert PAIRED["RRF", "dense", 3, "questions"]["only dense"] == ["d01-01"]
assert PAIRED["RRF", "dense", 3, "questions"]["only RRF"] == ["d03-13", "d07-a08"]
assert PAIRED["RRF", "BM25", 3, "questions"]["only RRF"] == ["d07-a08"] and PAIRED["RRF", "BM25", 3, "questions"]["neither"] == ["d01-01"]
assert len(PAIRED["RRF", "BM25", 1, "lookups"]["only BM25"]) == 10 and not PAIRED["RRF", "BM25", 1, "lookups"]["only RRF"]
assert sign_test(11, 0) < 0.001 and close(sign_test(10, 0), 0.002, 0.0001) and sign_test(2, 1) == 1.0
assert ranks("RRF", ["d01-01", "d04-v07"]) == [15, 2] and len(demoted) == 10 and len(bm25_hits) == 12
assert sorted(rrf_rank[c] for c in bm25_hits) == list(range(1, 13)) and dense_rank[bm25_hits[0]] == 39
assert [rrf_rank[c] for c, _ in rankings["dense"][DADS.id][:3]] == [13, 14, 15]
assert ranks("RRF", ["d03-13"])[0] in (2, 3) and ranks("RRF", ["d04-v10"])[0] in (1, 2)   # near ties in dense's input

# The weighted fusion (section 5)
assert ALPHA == 0.7 and sweep[0.7]["all dev"]["hit@3"] == 46
assert all(sweep[a]["lookups"]["hit@1"] == 30 for a in ALPHAS if a <= 0.7) and sweep[0.8]["lookups"]["hit@1"] < 30
assert all(sweep[a]["dev questions"]["hit@3"] == 16 for a in (0.7, 0.8)) and sweep[0.6]["dev questions"]["hit@3"] == 15
assert counts("weighted", "questions") == (27, 31, 31, 31) and counts("weighted", "lookups") == (30, 30, 30, 30)
test_part = [q for q in SELECTIONS["questions"] if q.split == "test"]
assert [measure(rankings[n], test_part)["hit@3"] for n in NAMES] == [14, 14, 15, 15]
assert [measure(rankings[n], test_part)["hit@1"] for n in NAMES] == [11, 13, 13, 14]
assert PAIRED["weighted", "RRF", 3, "questions"]["only weighted"] == ["d01-01"]
assert len(PAIRED["weighted", "RRF", 1, "lookups"]["only weighted"]) == 10

# Every miss (section 6): Day 07's 15, then one
assert len(misses) == 16 and [(n, q.id) for n, q in misses if n not in ("BM25", "dense")] == [("RRF", "d01-01")]

# Same text or not (section 7)
assert summary(texts["dense"]["no header"], SELECTIONS["questions"]).split()[:2] == ["27", "31"]
assert summary(texts["BM25"]["no header"], SELECTIONS["questions"]).split()[:2] == ["22", "28"]

# Permissions (section 8)
assert leaks == [] and {q.user for name in EXPOSED for q, _ in EXPOSED[name]} == {"alice"}
assert [len(EXPOSED[n]) for n in ("RRF", "weighted")] == [8, 8]
assert min(rank for _, (rank, _) in EXPOSED["RRF"]) == 4 and sum(rank <= 10 for _, (rank, _) in EXPOSED["RRF"]) == 4
assert min(rank for _, (rank, _) in EXPOSED["weighted"]) > 20

# Cost (section 9): the fusion is cheap next to embedding the question
assert timings["RRF fusion"][0] < timings["dense (embed the question + search)"][0]
assert timings["weighted fusion"][0] < timings["dense (embed the question + search)"][0]

# The baseline (section 10)
saved = json.loads(run_file.read_text(encoding="utf-8"))
assert saved["dataset"] == day07["dataset"] and len(saved["errors"]) == 16
assert saved["queries"]["d03-13"]["RRF"]["margin"] == 0.0001 and saved["queries"]["d04-v10"]["RRF"]["margin"] < 0.001
assert changes is None or changes == [], changes            # near ties (CPU rounding) are allowed
if BASELINE_FILE.is_file():                                  # and the same top 3 lists as the accepted run
    accepted = json.loads(BASELINE_FILE.read_text(encoding="utf-8"))
    assert all(saved["queries"][qid][n]["top3"] == accepted["queries"][qid][n]["top3"] for qid in by_qid for n in NAMES)

print("All Day 08 sanity checks passed.")

All Day 08 sanity checks passed.


## 12. Exercises

Each takes 5–15 minutes. Restart and rerun the notebook after changing anything.

1. **Weight RRF.** Azure AI Search and Elasticsearch let you weight each list in RRF. Give dense a weight of 2 (multiply its 1/(k + rank) by 2). What happens to new dads, and to the lookups at rank 1? Why can't a weight fix both?
2. **Another normalisation.** Replace `min_max()` with a z-score (subtract the mean, divide by the standard deviation, over the chunks each retriever returned). Rerun the α sweep on dev. Does the rule pick the same α, and does anything change on test?
3. **A shorter window.** Fuse only each retriever's top 5 (`rankings[name][q.id][:5]`) in both `rrf()` and `weighted()`. Which queries move? What does min-max mean when the window changes?
4. **Re-tune with the headerless vectors.** In section 7, run the α sweep with `texts["dense"]["no header"]` as the dense input. Does it pick the same α? Would you now change the decision to keep one text, and what would you need to see first?
5. **Feed the hybrid to Day 06's reader.** Build the context from the weighted fusion's top 3 (`assemble()` takes `Result` objects; see `rag_from_scratch.index.Result`) and run the `ExtractiveReader` on the 31 answerable questions. Does the wine question get "Alcohol is not reimbursed"? Which gate would you use, now that the fused score isn't a cosine similarity?